In [1]:
# ============================================================
# WEEK 4 — Noise Sensitivity Analysis & Threshold Tuning
# Project: Contextual Predictive Maintenance
# Intern Branch: preeti-dev
# Infotact Solutions
# ============================================================
 
 
# ── CELL 1: Install & Import Libraries ───────────────────
import subprocess, sys
 
def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])
 
for lib in ["pandas", "numpy", "matplotlib", "seaborn",
            "scikit-learn", "lightgbm", "imbalanced-learn"]:
    install(lib)
 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')
 
import lightgbm as lgb
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (f1_score, precision_recall_curve,
                             average_precision_score, confusion_matrix,
                             classification_report)
 
np.random.seed(42)
 
print("✅ All libraries imported successfully")

✅ All libraries imported successfully


In [2]:
# ── CELL 2: Create Folder Structure ──────────────────────
for folder in ['../data/raw', '../data/processed', '../reports']:
    os.makedirs(folder, exist_ok=True)
 
print("✅ Folder structure ready")

✅ Folder structure ready


In [4]:
# ── CELL 3: Load Week 2 Fused Dataset ────────────────────
df = pd.read_csv('../data/processed/week2_fused_features.csv')
 
print(f"✅ Dataset loaded")
print(f"   Rows    : {df.shape[0]}")
print(f"   Columns : {df.shape[1]}")

✅ Dataset loaded
   Rows    : 10000
   Columns : 35


In [5]:
# ── CELL 4: Define Features & Target ─────────────────────
drop_cols = ['UDI', 'Product ID', 'Type', 'timestamp',
             'Machine failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF']
drop_cols = [c for c in drop_cols if c in df.columns]
 
feature_cols = [c for c in df.columns if c not in drop_cols]
target       = 'Machine failure'
 
X = df[feature_cols]
y = df[target]
 
print(f"✅ Features defined : {len(feature_cols)} columns")
print(f"   Failure rate     : {y.mean()*100:.2f}%")


✅ Features defined : 25 columns
   Failure rate     : 3.39%


In [ ]:
# ── CELL 5: Train Final LightGBM Model ───────────────────
# We retrain the full model from Week 3 here so Week 4
# is a standalone notebook that doesn't depend on saved model files

# Rename features to remove special characters (brackets) that LightGBM doesn't support
X_renamed = X.copy()
X_renamed.columns = X_renamed.columns.str.replace('[', '_').str.replace(']', '')

# Apply SMOTE on training data only
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

# Ensure column names are preserved after SMOTE
X_train_sm.columns = X_train.columns
 
# Apply SMOTE on training data only
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)
 
lgb_params = {
    'objective'        : 'binary',
    'boosting_type'    : 'gbdt',
    'n_estimators'     : 500,
    'learning_rate'    : 0.05,
    'max_depth'        : 8,
    'num_leaves'       : 50,
    'min_child_samples': 20,
    'subsample'        : 0.8,
    'colsample_bytree' : 0.8,
    'scale_pos_weight' : (y==0).sum() / (y==1).sum(),
    'random_state'     : 42,
    'verbose'          : -1
}
 
model = lgb.LGBMClassifier(**lgb_params)
# ── CELL 5: Train Final LightGBM Model ───────────────────
# We retrain the full model from Week 3 here so Week 4
# is a standalone notebook that doesn't depend on saved model files

# Rename features to remove special characters (brackets) that LightGBM doesn't support
X_train = X_train.copy()
X_test  = X_test.copy()

X_train.columns = X_train.columns.str.replace('[', '_', regex=False).str.replace(']', '', regex=False)
X_test.columns  = X_test.columns.str.replace('[', '_', regex=False).str.replace(']', '', regex=False)

# Apply SMOTE on training data only
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

lgb_params = {
    'objective'        : 'binary',
    'boosting_type'    : 'gbdt',
    'n_estimators'     : 500,
    'learning_rate'    : 0.05,
    'max_depth'        : 8,
    'num_leaves'       : 50,
    'min_child_samples': 20,
    'subsample'        : 0.8,
    'colsample_bytree' : 0.8,
    'scale_pos_weight' : (y==0).sum() / (y==1).sum(),
    'random_state'     : 42,
    'verbose'          : -1
}

model = lgb.LGBMClassifier(**lgb_params)
model.fit(X_train_sm, y_train_sm)

# Baseline performance on clean test data (no noise)
y_proba_clean  = model.predict_proba(X_test)[:, 1]
y_pred_clean   = model.predict(X_test)
f1_clean       = f1_score(y_test, y_pred_clean, average='macro')

print(f"✅ Final LightGBM model trained")
print(f"   Baseline Macro F1 (clean data) : {f1_clean:.4f}")
 
# Baseline performance on clean test data (no noise)
y_proba_clean  = model.predict_proba(X_test)[:, 1]
y_pred_clean   = model.predict(X_test)
f1_clean       = f1_score(y_test, y_pred_clean, average='macro')
 
print(f"✅ Final LightGBM model trained")
print(f"   Baseline Macro F1 (clean data) : {f1_clean:.4f}")

LightGBMError: Do not support special JSON characters in feature name.